# Donut fine-tune for InBody extraction — Colab runner

Runs `cera.training.train` on Colab GPU instead of the local 8GB laptop GPU (which is compute-bound on `donut-base`'s 2560x1920 canvas — ~150s/step locally).

**Before running**: Runtime -> Change runtime type -> pick a GPU (T4 is free tier; A100 needs Colab Pro and is much faster).

Two ways to get the 5,000-sheet synthetic dataset onto Colab (pick one in the cell below):
- **Regenerate** (default, no manual step): installs headless Chromium and re-runs the same seeded generator used locally — reproducible, ~60-90 min, runs before training starts.
- **Upload a zip**: if you already zipped `D:\cera\data\synthetic` and uploaded it to your Drive, mount Drive and point `DATA_ZIP` at it — skips regeneration.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
GITHUB_REPO = "https://github.com/QeekOw/CERA.git"  # if private, use a token: https://<TOKEN>@github.com/QeekOw/CERA.git
BRANCH = "feat/ocr-inbody-extraction"

!git clone --branch $BRANCH --single-branch $GITHUB_REPO repo
%cd repo
!pip install -q -e ".[training]"

In [ ]:
USE_ZIP = False  # flip to True and set DATA_ZIP if you uploaded a dataset zip to Drive instead
DATA_ZIP = "/content/drive/MyDrive/cera/synthetic.zip"

if USE_ZIP:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/data/synthetic
    !unzip -q "$DATA_ZIP" -d /content/data/synthetic
else:
    !apt-get -qq install -y chromium-browser > /dev/null
    !python -c "from pathlib import Path; from cera.training.dataset import generate_dataset; generate_dataset(Path('/content/data/synthetic'))"

DATA_DIR = "/content/data/synthetic"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CHECKPOINT_DIR = "/content/drive/MyDrive/cera/checkpoints/donut-inbody"  # persists past session timeout
!mkdir -p "$CHECKPOINT_DIR"

In [ ]:
# T4 (16GB): batch 2, no checkpointing needed. A100 (40GB): bump batch to 4-8.
!python -m cera.training.train \
  --data-dir "$DATA_DIR" --output-dir "$CHECKPOINT_DIR" \
  --model-name-or-path naver-clova-ix/donut-base \
  --epochs 3 --batch-size 2 --gradient-accumulation-steps 2 --learning-rate 3e-5

If disconnected mid-run: re-run this notebook from the top, then re-run the training cell with `--resume` appended — checkpoints save every epoch to Drive, so nothing before the last completed epoch is lost.